In [ ]:
import os
import sys

# Repo root whether the notebook cwd is <repo>/tacotron2 or <repo>/
_cwd = os.getcwd()
if os.path.basename(_cwd.rstrip("/\\")) == "tacotron2":
    _REPO_ROOT = os.path.dirname(_cwd)
else:
    _REPO_ROOT = _cwd
for _p in (_REPO_ROOT, os.path.join(_REPO_ROOT, "tacotron2")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import torch
import matplotlib.pyplot as plt
from IPython.display import Audio

import numpy as np
import soundfile as sf
import torchaudio

from commons.dataset import AudioMelConversions
from commons.hyperparams import Tacotron2Config
from model import Tacotron2
from tokenizer import Tokenizer

# Notebook-style checkpoint: model_state_dict + config (tries common layouts)
# Prefer epoch 44 (epoch 45 file appears corrupted in this workspace).
_CKPT_CANDIDATES = [
    os.path.join(_REPO_ROOT, "checkpoints_taco2", "tacotron2_epoch_0044.pth"),
    os.path.join(_REPO_ROOT, "tacotron2-wavernn", "checkpoints_taco2", "tacotron2_epoch_0044.pth"),
    os.path.join(_REPO_ROOT, "checkpoints_taco2", "tacotron2_epoch_0045.pth"),
    os.path.join(_REPO_ROOT, "tacotron2-wavernn", "checkpoints_taco2", "tacotron2_epoch_0045.pth"),
]
CHECKPOINT_PATH = next(
    (os.path.normpath(p) for p in _CKPT_CANDIDATES if os.path.isfile(p)), None
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GRIFFIN_LIM_ITERS = 60


def _load_checkpoint(path: str, map_location=None):
    """Unpickle onto CPU first (reliable on Colab); map_location is unused."""
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        head = f.read(120)
    if size < 20_000:
        if b"git-lfs.github.com" in head or b"version https://git-lfs" in head:
            raise RuntimeError(
                "This file is a Git LFS pointer, not the real checkpoint. "
                "Run `git lfs pull` or copy the full .pth from the machine that trained."
            )
        raise RuntimeError(
            f"Checkpoint is only {size} bytes — too small for a Tacotron2 .pth. "
            "Point CHECKPOINT_PATH at the full weights file (often hundreds of MB)."
        )

    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        ck = torch.load(path, map_location="cpu")
    except RuntimeError as e:
        err = str(e).lower()
        if "zip" in err or "central directory" in err or "pytorchstreamreader" in err:
            raise RuntimeError(
                "torch.load could not read this .pth — the file is usually truncated or corrupted.\n"
                f"  Path: {path}\n"
                f"  Size on disk: {size / (1024 * 1024):.2f} MiB\n"
                "Typical causes: incomplete upload/sync, or a bad copy.\n"
                "Fix: re-copy the full checkpoint and compare file sizes."
            ) from e
        raise

    if isinstance(ck, dict) and "model_state_dict" in ck:
        return ck.get("config"), ck["model_state_dict"]
    return None, ck


if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        "No checkpoint found. Tried:\n"
        + "\n".join(f"  - {os.path.normpath(p)}" for p in _CKPT_CANDIDATES)
        + "\nSet CHECKPOINT_PATH to your .pth file explicitly."
    )

saved_config, state_dict = _load_checkpoint(CHECKPOINT_PATH, DEVICE)
config = saved_config if saved_config is not None else Tacotron2Config()
model = Tacotron2(config).to(DEVICE)
model.load_state_dict(state_dict, strict=True)
model.eval()

tokenizer = Tokenizer()
a2m = AudioMelConversions(
    num_mels=config.num_mels,
    sampling_rate=config.sample_rate,
    n_fft=config.n_fft,
    window_size=config.win_length,
    hop_size=config.hop_length,
    fmin=config.fmin,
    fmax=config.fmax,
    min_db=config.min_db,
    max_scaled_abs=config.max_scaled_abs,
)
print(f"Loaded {CHECKPOINT_PATH} on {DEVICE}")


def inference(
    text,
    griffin_lim_iters=GRIFFIN_LIM_ITERS,
    max_decode_steps=2000,
    save_path=None,
):
    """Synthesize with Tacotron2 + Griffin–Lim; show mel, attention, play audio, optionally save WAV."""
    print(f"Input text: {text!r}")
    tokens = tokenizer.encode(text).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        mel_post, alignments = model.inference(tokens, max_decode_steps=max_decode_steps)

    fig, axes = plt.subplots(2, 1, figsize=(6, 7))
    im0 = axes[0].imshow(
        mel_post[0].T.cpu().numpy(), aspect="auto", origin="lower", interpolation="none"
    )
    axes[0].set_title("Predicted mel (post-net)")
    axes[0].set_ylabel("Mel bin")
    fig.colorbar(im0, ax=axes[0])

    im1 = axes[1].imshow(
        alignments[0].T.cpu().numpy(), aspect="auto", origin="lower", interpolation="none"
    )
    axes[1].set_title("Attention alignment")
    axes[1].set_xlabel("Decoder time")
    axes[1].set_ylabel("Encoder position")
    fig.colorbar(im1, ax=axes[1])
    plt.tight_layout()
    plt.show()

    audio_i16 = a2m.mel2audio(
        mel_post[0].T.cpu(), do_denorm=True, griffin_lim_iters=griffin_lim_iters
    )

    # Prefer float audio for notebook playback.
    audio_f32 = audio_i16.astype(np.float32) / 32768.0

    # If decoding hits max steps, we often get a long low-energy tail.
    # Trim near-silence and normalize to a reasonable peak for easier listening.
    thr = 1e-3
    idx = np.where(np.abs(audio_f32) > thr)[0]
    if idx.size > 0:
        audio_f32 = audio_f32[idx[0] : idx[-1] + 1]
    peak = float(np.max(np.abs(audio_f32))) if audio_f32.size else 0.0
    if peak > 0:
        audio_f32 = 0.95 * (audio_f32 / peak)

    if save_path:
        save_path = os.path.normpath(os.path.join(_REPO_ROOT, save_path))
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        sf.write(save_path, audio_f32, config.sample_rate)
        print(f"Saved audio to: {save_path}")

    display(Audio(audio_f32, rate=config.sample_rate))
    return mel_post, alignments, audio_i16


In [ ]:
inference(
    "شكرا جزيلا لاستماعكم",
    save_path=os.path.join("outputs", "tacotron2_epoch44_sample.wav"),
)
